# Data cleaning toàn bộ 14 file nguồn

Notebook này làm sạch toàn bộ dataset ở grain của từng file, không ghi đè CSV nguồn và không xóa outlier. Quy trình gồm: profiling, kiểm tra missing, duplicate, chuẩn hóa kiểu/giá trị, gắn cờ outlier, kiểm tra nghiệp vụ, kiểm tra toàn vẹn và đối chiếu kết quả.

Phạm vi là 14 file nguồn chính thức, tổng cộng **2.960.736 dòng**. `data/submission.csv` là output của mô hình nên không thuộc phạm vi cleaning. Output được ghi riêng vào `data/silver/full_dataset_notebook/`.

## 1. Câu chuyện dữ liệu và grain

- `geography → customers → orders → order_items ← products`;
- payment và shipment ở grain đơn hàng; return và review ở grain dòng hàng;
- inventory ở grain sản phẩm–snapshot tháng; web traffic và sales ở grain ngày;
- `sample_submission` là 548 ngày tương lai cần dự báo.

`order_items` không có khóa dòng hàng từ nguồn. Notebook tái dựng `line_number` theo thứ tự vật lý trong từng `order_id`. Return/review được ánh xạ theo occurrence; đây là giả định tái dựng có công bố, không phải ground truth từ hệ thống nguồn.

In [1]:
from pathlib import Path
import gc
import hashlib
import os
import tempfile

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 80)

DATA_DIR = Path('data')
OUTPUT_DIR = DATA_DIR / 'silver' / 'full_dataset_notebook'
CLEANING_VERSION = '1.0.0'
TOTAL_EXPECTED_ROWS = 2_960_736

EXPECTED_COLUMNS = {
    'customers': ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel'],
    'geography': ['zip', 'city', 'region', 'district'],
    'inventory': ['snapshot_date', 'product_id', 'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate', 'stockout_flag', 'overstock_flag', 'reorder_flag', 'sell_through_rate', 'product_name', 'category', 'segment', 'year', 'month'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount', 'promo_id', 'promo_id_2'],
    'orders': ['order_id', 'order_date', 'customer_id', 'zip', 'order_status', 'payment_method', 'device_type', 'order_source'],
    'payments': ['order_id', 'payment_method', 'payment_value', 'installments'],
    'products': ['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs'],
    'promotions': ['promo_id', 'promo_name', 'promo_type', 'discount_value', 'start_date', 'end_date', 'applicable_category', 'promo_channel', 'stackable_flag', 'min_order_value'],
    'returns': ['return_id', 'order_id', 'product_id', 'return_date', 'return_reason', 'return_quantity', 'refund_amount'],
    'reviews': ['review_id', 'order_id', 'product_id', 'customer_id', 'review_date', 'rating', 'review_title'],
    'sales': ['Date', 'Revenue', 'COGS'],
    'sample_submission': ['Date', 'Revenue', 'COGS'],
    'shipments': ['order_id', 'ship_date', 'delivery_date', 'shipping_fee'],
    'web_traffic': ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source'],
}
EXPECTED_ROWS = {
    'customers': 121_930, 'geography': 39_948, 'inventory': 60_247,
    'order_items': 714_669, 'orders': 646_945, 'payments': 646_945,
    'products': 2_412, 'promotions': 50, 'returns': 39_939,
    'reviews': 113_551, 'sales': 3_833, 'sample_submission': 548,
    'shipments': 566_067, 'web_traffic': 3_652,
}
OPTIONAL_COLUMNS = {
    'customers': {'signup_date'},
    'order_items': {'promo_id', 'promo_id_2'},
    'promotions': {'applicable_category'},
    'shipments': {'delivery_date'},
}
DATE_COLUMNS = {
    'customers': ['signup_date'], 'inventory': ['snapshot_date'],
    'orders': ['order_date'], 'promotions': ['start_date', 'end_date'],
    'returns': ['return_date'], 'reviews': ['review_date'],
    'sales': ['Date'], 'sample_submission': ['Date'],
    'shipments': ['ship_date', 'delivery_date'], 'web_traffic': ['date'],
}
INTEGER_COLUMNS = {
    'customers': ['customer_id', 'zip'], 'geography': ['zip'],
    'inventory': ['product_id', 'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'year', 'month'],
    'order_items': ['order_id', 'product_id', 'quantity'],
    'orders': ['order_id', 'customer_id', 'zip'], 'payments': ['order_id', 'installments'],
    'products': ['product_id'], 'promotions': ['min_order_value'],
    'returns': ['order_id', 'product_id', 'return_quantity'],
    'reviews': ['order_id', 'product_id', 'customer_id', 'rating'],
    'shipments': ['order_id'],
    'web_traffic': ['sessions', 'unique_visitors', 'page_views'],
}
FLOAT_COLUMNS = {
    'inventory': ['days_of_supply', 'fill_rate', 'sell_through_rate'],
    'order_items': ['unit_price', 'discount_amount'],
    'payments': ['payment_value'], 'products': ['price', 'cogs'],
    'promotions': ['discount_value'], 'returns': ['refund_amount'],
    'sales': ['Revenue', 'COGS'], 'sample_submission': ['Revenue', 'COGS'],
    'shipments': ['shipping_fee'],
    'web_traffic': ['bounce_rate', 'avg_session_duration_sec'],
}
BOOLEAN_COLUMNS = {
    'inventory': ['stockout_flag', 'overstock_flag', 'reorder_flag'],
    'promotions': ['stackable_flag'],
}
PRIMARY_KEYS = {
    'customers': ['customer_id'], 'geography': ['zip'],
    'inventory': ['snapshot_date', 'product_id'],
    'orders': ['order_id'], 'payments': ['order_id'], 'products': ['product_id'],
    'promotions': ['promo_id'], 'returns': ['return_id'], 'reviews': ['review_id'],
    'sales': ['Date'], 'sample_submission': ['Date'], 'shipments': ['order_id'],
    'web_traffic': ['date'],
}
DOMAINS = {
    ('customers', 'gender'): {'Female', 'Male', 'Non-binary'},
    ('customers', 'age_group'): {'18-24', '25-34', '35-44', '45-54', '55+'},
    ('customers', 'acquisition_channel'): {'organic_search', 'social_media', 'paid_search', 'email_campaign', 'referral', 'direct'},
    ('geography', 'region'): {'East', 'Central', 'West'},
    ('products', 'category'): {'Streetwear', 'Outdoor', 'Casual', 'GenZ'},
    ('products', 'segment'): {'Activewear', 'Everyday', 'Performance', 'Balanced', 'Standard', 'Premium', 'All-weather', 'Trendy'},
    ('products', 'size'): {'S', 'M', 'L', 'XL'},
    ('orders', 'order_status'): {'delivered', 'cancelled', 'returned', 'shipped', 'paid', 'created'},
    ('orders', 'payment_method'): {'credit_card', 'paypal', 'cod', 'apple_pay', 'bank_transfer'},
    ('orders', 'device_type'): {'mobile', 'desktop', 'tablet'},
    ('orders', 'order_source'): {'organic_search', 'paid_search', 'social_media', 'email_campaign', 'referral', 'direct'},
    ('payments', 'payment_method'): {'credit_card', 'paypal', 'cod', 'apple_pay', 'bank_transfer'},
    ('promotions', 'promo_type'): {'percentage', 'fixed'},
    ('promotions', 'promo_channel'): {'all_channels', 'online', 'email', 'social_media', 'in_store'},
    ('returns', 'return_reason'): {'wrong_size', 'defective', 'not_as_described', 'changed_mind', 'late_delivery'},
    ('web_traffic', 'traffic_source'): {'organic_search', 'paid_search', 'social_media', 'email_campaign', 'referral', 'direct'},
}
OUTLIER_COLUMNS = {
    'products': ['price', 'cogs'], 'promotions': ['discount_value', 'min_order_value'],
    'order_items': ['quantity', 'discount_amount'], 'payments': ['payment_value'],
    'shipments': ['shipping_fee'], 'returns': ['return_quantity', 'refund_amount'],
    'inventory': ['stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate', 'sell_through_rate'],
    'web_traffic': ['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec'],
    'sales': ['Revenue', 'COGS'], 'sample_submission': ['Revenue', 'COGS'],
}
print(f'Phạm vi: {len(EXPECTED_COLUMNS)} file, {sum(EXPECTED_ROWS.values()):,} dòng mong đợi.')

Phạm vi: 14 file, 2,960,736 dòng mong đợi.


### Nhận xét và kết luận

- Cell đã khai báo contract cho **14 file nguồn** với tổng số dòng mong đợi là **2.960.736**. Contract bao gồm header, row count, cột optional, kiểu dữ liệu, khóa, domain và các cột cần kiểm tra outlier.
- `submission.csv` không nằm trong danh sách vì đây là output của mô hình, không phải dữ liệu nguồn.
- Đây mới là cấu hình kỳ vọng, chưa phải bằng chứng rằng dữ liệu thực tế đạt contract. Bằng chứng được tạo ở các cell đọc dữ liệu và acceptance phía sau.

**Kết luận:** phạm vi cleaning đã được khóa rõ ràng trước khi biến đổi, giúp phát hiện ngay khi file hoặc schema nguồn thay đổi ngoài dự kiến.

## 2. Hàm dùng chung

Các hàm bên dưới áp dụng fail-fast: sai header, row count, giá trị bắt buộc, kiểu dữ liệu hoặc domain thì notebook dừng trước khi ghi output.

In [2]:
class DataQualityError(ValueError):
    pass

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def parse_integer(series, dataset, column):
    numeric = pd.to_numeric(series, errors='coerce')
    invalid = series.notna() & (numeric.isna() | numeric.mod(1).ne(0))
    if invalid.any():
        raise DataQualityError(f'{dataset}.{column}: giá trị không phải số nguyên; ví dụ {series[invalid].head().tolist()}')
    return numeric.astype('Int64')

def parse_number(series, dataset, column):
    numeric = pd.to_numeric(series, errors='coerce')
    invalid = series.notna() & numeric.isna()
    if invalid.any():
        raise DataQualityError(f'{dataset}.{column}: giá trị không phải số; ví dụ {series[invalid].head().tolist()}')
    return numeric.astype('Float64')

def parse_date(series, dataset, column):
    parsed = pd.to_datetime(series, format='%Y-%m-%d', errors='coerce')
    invalid = series.notna() & parsed.isna()
    if invalid.any():
        raise DataQualityError(f'{dataset}.{column}: ngày không đúng YYYY-MM-DD; ví dụ {series[invalid].head().tolist()}')
    return parsed

def parse_boolean(series, dataset, column):
    numeric = parse_integer(series, dataset, column)
    invalid = numeric.notna() & ~numeric.isin([0, 1])
    if invalid.any():
        raise DataQualityError(f'{dataset}.{column}: boolean chỉ nhận 0/1.')
    return numeric.map({0: False, 1: True}).astype('boolean')

def profile_column(dataset, column, series, raw_missing):
    result = {
        'dataset': dataset, 'column': column, 'rows': len(series),
        'missing_count': int(raw_missing), 'missing_pct': raw_missing / len(series),
        'distinct_count': int(series.nunique(dropna=True)),
        'logical_type': str(series.dtype), 'min_value': None, 'q1': None,
        'median': None, 'q3': None, 'max_value': None,
    }
    if pd.api.types.is_datetime64_any_dtype(series):
        valid = series.dropna()
        if len(valid):
            result['min_value'] = valid.min().date().isoformat()
            result['max_value'] = valid.max().date().isoformat()
    elif pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
        valid = pd.to_numeric(series.dropna(), errors='raise').astype('float64')
        if len(valid):
            result.update({
                'min_value': float(valid.min()), 'q1': float(valid.quantile(0.25)),
                'median': float(valid.median()), 'q3': float(valid.quantile(0.75)),
                'max_value': float(valid.max()),
            })
    return result

def orphan_count(child, child_columns, parent, parent_columns):
    left = child[list(child_columns)].dropna().drop_duplicates()
    right = parent[list(parent_columns)].drop_duplicates()
    right = right.rename(columns=dict(zip(parent_columns, child_columns)))
    joined = left.merge(right, on=list(child_columns), how='left', indicator=True)
    return int(joined['_merge'].eq('left_only').sum())

quality_records = []
def record_check(category, dataset, check, violations, details, severity='FAIL'):
    quality_records.append({
        'category': category, 'dataset': dataset, 'check': check,
        'severity': severity, 'violations': int(violations),
        'status': 'PASS' if int(violations) == 0 else ('WARN' if severity == 'WARN' else 'FAIL'),
        'details': details,
    })

def record_mask(category, dataset, check, valid_mask, details, severity='FAIL'):
    valid = pd.Series(valid_mask).fillna(False)
    record_check(category, dataset, check, int((~valid).sum()), details, severity)

### Nhận xét và kết luận

- Cell này chỉ định nghĩa các hàm dùng chung: checksum SHA-256, parse integer/number/date/boolean, profiling, đếm orphan và ghi nhận quality check. Vì vậy cell không có bảng kết quả là đúng thiết kế.
- Cách parse là **fail-fast**: giá trị không chuyển đổi được sẽ làm notebook dừng thay vì âm thầm biến thành null.
- `record_check` tách `FAIL` khỏi `WARN`, giúp phân biệt lỗi phá vỡ contract với bất thường chỉ cần review.

**Kết luận:** cell tạo nền tảng kiểm tra có thể tái sử dụng; chưa thể kết luận chất lượng dữ liệu chỉ từ cell khai báo hàm này.

## 3. Data profiling, missing và chuẩn hóa kiểu

Mỗi file được đọc dưới dạng chuỗi để kiểm tra raw contract trước, sau đó trim, chuyển ô trống optional thành null và parse sang integer, number, boolean hoặc date. Notebook tạo profile theo từng cột gồm missing, distinct, min, Q1, median, Q3 và max.

In [3]:
cleaned = {}
profile_records = []
source_audit_records = []
source_hashes = {}

for dataset, expected_columns in EXPECTED_COLUMNS.items():
    path = DATA_DIR / f'{dataset}.csv'
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy {path}; tải data theo README.md.')
    raw = pd.read_csv(
        path, dtype='string', keep_default_na=False, na_filter=False,
        encoding='utf-8-sig', low_memory=False,
    )
    if list(raw.columns) != expected_columns:
        raise DataQualityError(f'{dataset}: header sai: {list(raw.columns)}')
    if len(raw) != EXPECTED_ROWS[dataset]:
        raise DataQualityError(f'{dataset}: row count {len(raw):,} != {EXPECTED_ROWS[dataset]:,}')

    for column in raw.columns:
        raw[column] = raw[column].str.strip()
    missing_by_column = {column: int(raw[column].eq('').sum()) for column in raw.columns}
    optional = OPTIONAL_COLUMNS.get(dataset, set())
    required_missing = sum(count for column, count in missing_by_column.items() if column not in optional)
    duplicate_full_rows = int(raw.duplicated().sum())
    record_check('missing', dataset, 'required_values_present', required_missing, 'Không được thiếu cột bắt buộc.')
    record_check('duplicate', dataset, 'no_full_row_duplicates', duplicate_full_rows, 'Không tự drop duplicate; fail nếu có duplicate toàn dòng.')
    if required_missing:
        raise DataQualityError(f'{dataset}: có {required_missing} ô trống bắt buộc.')
    if duplicate_full_rows:
        raise DataQualityError(f'{dataset}: có {duplicate_full_rows} duplicate toàn dòng cần quyết định nghiệp vụ.')

    frame = raw.mask(raw.eq(''), pd.NA)
    for column in DATE_COLUMNS.get(dataset, []):
        frame[column] = parse_date(frame[column], dataset, column)
    for column in INTEGER_COLUMNS.get(dataset, []):
        frame[column] = parse_integer(frame[column], dataset, column)
    for column in FLOAT_COLUMNS.get(dataset, []):
        frame[column] = parse_number(frame[column], dataset, column)
    for column in BOOLEAN_COLUMNS.get(dataset, []):
        frame[column] = parse_boolean(frame[column], dataset, column)
    typed = set(DATE_COLUMNS.get(dataset, [])) | set(INTEGER_COLUMNS.get(dataset, [])) | set(FLOAT_COLUMNS.get(dataset, [])) | set(BOOLEAN_COLUMNS.get(dataset, []))
    for column in set(frame.columns) - typed:
        frame[column] = frame[column].astype('string[pyarrow]')

    for column in frame.columns:
        profile_records.append(profile_column(dataset, column, frame[column], missing_by_column[column]))
    frame.insert(0, 'source_row_number', pd.Series(np.arange(1, len(frame) + 1), dtype='Int64'))
    cleaned[dataset] = frame
    source_hashes[dataset] = sha256_file(path)
    source_audit_records.append({
        'dataset': dataset, 'rows': len(frame), 'source_columns': len(expected_columns),
        'required_missing': required_missing, 'optional_missing': sum(missing_by_column[c] for c in optional),
        'duplicate_full_rows': duplicate_full_rows, 'sha256': source_hashes[dataset],
    })
    del raw
    gc.collect()

profile_summary = pd.DataFrame(profile_records)
for column in ['min_value', 'max_value']:
    profile_summary[column] = profile_summary[column].astype('string[pyarrow]')
source_audit = pd.DataFrame(source_audit_records)
assert int(source_audit['rows'].sum()) == TOTAL_EXPECTED_ROWS
display(source_audit)
display(profile_summary.head(20))

,dataset,rows,source_columns,required_missing,optional_missing,duplicate_full_rows,sha256
0,customers,121930,7,0,0,0,5e58abf49e99d52bf8ddda44601a30e78667760908b862...
1,geography,39948,4,0,0,0,f5d150bdfdfca7ac0c38118b1f5f8ea1b3f5de4cd23fa7...
2,inventory,60247,17,0,0,0,1bbcdd89569a5df4b938ac0577f0657030c41c3a18956f...
3,order_items,714669,7,0,1152816,0,a8b2adfe54eec0dda79ced28e5987d6846b7f31cb60523...
4,orders,646945,8,0,0,0,f4b3029f386f5f5a6baabd78aad0140e5a101ea6975f4c...
5,payments,646945,4,0,0,0,aacef7716fe1c1442bc46e8885019cf1d67e12655c2e0f...
6,products,2412,8,0,0,0,890946808d8237dad0d39153efd427f5399a5e6eaa2302...
7,promotions,50,10,0,40,0,158ee5843ee41a28b5f5748ab2c73bd712524da6c00930...
8,returns,39939,7,0,0,0,501717bc76cf237c0dbbb68f586351870047198042c423...
9,reviews,113551,7,0,0,0,5230859408955d611bc430df5d788eb66fedd56f3be9f6...


,dataset,column,rows,missing_count,missing_pct,distinct_count,logical_type,min_value,q1,median,q3,max_value
0,customers,customer_id,121930,0,0.0,121930,Int64,1.0,39343.5000,78784.5000,118156.75,157563.0
1,customers,zip,121930,0,0.0,31491,Int64,1001.0,28689.2500,49835.0000,73488.00,99950.0
2,customers,city,121930,0,0.0,42,string,<NA>,NaN,NaN,NaN,<NA>
3,customers,signup_date,121930,0,0.0,3941,datetime64[us],2012-01-17,NaN,NaN,NaN,2022-12-31
4,customers,gender,121930,0,0.0,3,string,<NA>,NaN,NaN,NaN,<NA>
5,customers,age_group,121930,0,0.0,5,string,<NA>,NaN,NaN,NaN,<NA>
6,customers,acquisition_channel,121930,0,0.0,6,string,<NA>,NaN,NaN,NaN,<NA>
7,geography,zip,39948,0,0.0,39948,Int64,1.0,28279.5000,49876.5000,73526.25,99950.0
8,geography,city,39948,0,0.0,42,string,<NA>,NaN,NaN,NaN,<NA>
9,geography,region,39948,0,0.0,3,string,<NA>,NaN,NaN,NaN,<NA>


### Nhận xét và kết luận

- Cả **14 file** khớp header và row count mong đợi; tổng cộng **2.960.736 dòng**. Mỗi file có SHA-256 để nhận diện chính xác snapshot đã xử lý.
- Không có missing ở cột bắt buộc và không có duplicate toàn dòng. Do đó không cần impute hoặc drop bản ghi.
- Missing chỉ xuất hiện ở trường optional: `order_items.promo_id` có **438.353** ô trống, `promo_id_2` có **714.463** ô trống và `promotions.applicable_category` có **40** ô trống. Con số `1.152.816` của `order_items` là tổng số **ô** optional trống, không phải số dòng bị thiếu.
- Profile đã bao phủ **96 cột**, gồm type, missing, cardinality, min, Q1, median, Q3 và max. Các phân vị là mô tả snapshot, không phải ngưỡng nghiệp vụ cố định.

**Kết luận:** dữ liệu nguồn đầy đủ về cấu trúc và trường bắt buộc; hành động cleaning phù hợp là chuẩn hóa kiểu và đổi chuỗi rỗng optional thành null, không tạo hoặc xóa dữ liệu.

## 4. Duplicate, khóa và ánh xạ dòng hàng

Duplicate toàn dòng là lỗi. Riêng 16 cặp `(order_id, product_id)` lặp không được xóa vì đó là 32 dòng hàng thật. Notebook sinh `line_number`, gom hai cột promotion thành `promo_ids`, rồi ánh xạ return/review theo occurrence.

In [4]:
items = cleaned['order_items']
items['line_number'] = items.groupby('order_id', sort=False).cumcount().add(1).astype('Int64')
items['product_occurrence'] = items.groupby(['order_id', 'product_id'], sort=False).cumcount().add(1).astype('Int64')
pair_counts = items.groupby(['order_id', 'product_id'], sort=False).size()
repeated_pairs = pair_counts[pair_counts > 1]
record_check('duplicate', 'order_items', 'repeated_order_product_pairs_preserved', abs(len(repeated_pairs) - 16) + abs(int(repeated_pairs.sum()) - 32), 'Mong đợi giữ 16 cặp / 32 dòng, không drop.')
items['promo_ids'] = [[value for value in pair if pd.notna(value)] for pair in zip(items['promo_id'], items['promo_id_2'])]
items.drop(columns=['promo_id', 'promo_id_2'], inplace=True)

item_map = items[['order_id', 'product_id', 'product_occurrence', 'line_number', 'quantity']].copy()
for dataset in ['returns', 'reviews']:
    frame = cleaned[dataset]
    frame['product_occurrence'] = frame.groupby(['order_id', 'product_id'], sort=False).cumcount().add(1).astype('Int64')
    mapped = frame.merge(
        item_map, on=['order_id', 'product_id', 'product_occurrence'],
        how='left', sort=False, validate='one_to_one', suffixes=('', '_ordered'),
    ).sort_values('source_row_number', kind='stable').reset_index(drop=True)
    record_check('integrity', dataset, 'occurrence_maps_to_order_item', int(mapped['line_number'].isna().sum()), 'Occurrence phải ánh xạ đúng một line_number.')
    if dataset == 'returns':
        mapped.rename(columns={'quantity': 'ordered_quantity'}, inplace=True)
    else:
        mapped.drop(columns=['quantity'], inplace=True)
    cleaned[dataset] = mapped

for dataset, key_columns in PRIMARY_KEYS.items():
    duplicates = int(cleaned[dataset].duplicated(key_columns).sum())
    record_check('duplicate', dataset, f'unique_key_{key_columns}', duplicates, 'Khóa nguồn phải unique.')
record_check('duplicate', 'order_items', 'unique_key_order_id_line_number', int(items.duplicated(['order_id', 'line_number']).sum()), 'Khóa tái dựng phải unique.')

promo_links = items[['order_id', 'line_number', 'promo_ids']].explode('promo_ids').dropna(subset=['promo_ids']).rename(columns={'promo_ids': 'promo_id'})
record_check('duplicate', 'order_items', 'unique_promotion_links', int(promo_links.duplicated(['order_id', 'line_number', 'promo_id']).sum()), 'Mỗi promotion chỉ xuất hiện một lần trên một dòng hàng.')
record_check('reconciliation', 'order_items', 'promotion_association_count', abs(len(promo_links) - 276_522), 'Mong đợi 276.522 liên kết promotion.')
record_check('reconciliation', 'order_items', 'two_promotion_row_count', abs(int(items['promo_ids'].map(len).eq(2).sum()) - 206), 'Mong đợi 206 dòng có hai promotion.')
print(f'Giữ {len(repeated_pairs)} cặp lặp / {int(repeated_pairs.sum())} dòng; tạo {len(promo_links):,} liên kết promotion.')

Giữ 16 cặp lặp / 32 dòng; tạo 276,522 liên kết promotion.


### Nhận xét và kết luận

- Có **16 cặp `(order_id, product_id)` lặp**, tương ứng **32 dòng hàng**. Các dòng này được giữ nguyên vì khác quantity hoặc unit_price; drop chúng sẽ làm sai Revenue và COGS.
- Khóa tái dựng `(order_id, line_number)` là unique. `line_number` được đánh theo thứ tự vật lý trong từng order trước khi join dữ liệu phụ thuộc.
- Hai cột promotion được gom thành `promo_ids`, bảo toàn **276.522 liên kết promotion**; trong đó **206 dòng hàng có hai promotion**.
- Return và review được map sang `line_number` theo occurrence của `(order_id, product_id)`. Đây là giả định tái dựng có thể lặp lại, không phải khóa ground truth do nguồn cung cấp.

**Kết luận:** notebook xử lý trùng lặp theo đúng grain, giữ đủ giao dịch và tạo khóa an toàn cho các quan hệ downstream.

## 5. Quy tắc nghiệp vụ và toàn vẹn tham chiếu

Các kiểm tra FAIL bảo vệ domain, số dương/không âm, thứ tự ngày, FK và các công thức có thể tái tạo. Những pattern đáng ngờ nhưng chưa đủ cơ sở sửa được ghi WARN.

In [5]:
# Domain
for (dataset, column), allowed in DOMAINS.items():
    invalid = int((cleaned[dataset][column].notna() & ~cleaned[dataset][column].isin(allowed)).sum())
    record_check('business_rule', dataset, f'{column}_domain', invalid, f'Giá trị hợp lệ: {sorted(allowed)}')

# FK đơn và FK promotion
fk_specs = [
    ('customers', ['zip'], 'geography', ['zip']),
    ('orders', ['customer_id'], 'customers', ['customer_id']), ('orders', ['zip'], 'geography', ['zip']),
    ('order_items', ['order_id'], 'orders', ['order_id']), ('order_items', ['product_id'], 'products', ['product_id']),
    ('payments', ['order_id'], 'orders', ['order_id']), ('shipments', ['order_id'], 'orders', ['order_id']),
    ('returns', ['order_id'], 'orders', ['order_id']), ('returns', ['product_id'], 'products', ['product_id']),
    ('reviews', ['order_id'], 'orders', ['order_id']), ('reviews', ['product_id'], 'products', ['product_id']),
    ('reviews', ['customer_id'], 'customers', ['customer_id']), ('inventory', ['product_id'], 'products', ['product_id']),
]
for child_name, child_columns, parent_name, parent_columns in fk_specs:
    count = orphan_count(cleaned[child_name], child_columns, cleaned[parent_name], parent_columns)
    record_check('integrity', child_name, f"fk_{'_'.join(child_columns)}_to_{parent_name}", count, 'Không được có orphan.')
promo_orphans = int((~promo_links['promo_id'].isin(cleaned['promotions']['promo_id'])).sum())
record_check('integrity', 'order_items', 'fk_promo_ids_to_promotions', promo_orphans, 'Mọi promo_id phải tồn tại trong promotions.')

# Quy tắc số và ngày
products = cleaned['products']; orders = cleaned['orders']; payments = cleaned['payments']
promotions = cleaned['promotions']; shipments = cleaned['shipments']; returns = cleaned['returns']
reviews = cleaned['reviews']; inventory = cleaned['inventory']; traffic = cleaned['web_traffic']
sales = cleaned['sales']; forecast = cleaned['sample_submission']; customers = cleaned['customers']; geography = cleaned['geography']
record_mask('business_rule', 'products', 'price_positive', products['price'] > 0, 'price phải > 0.')
record_mask('business_rule', 'products', 'cogs_nonnegative', products['cogs'] >= 0, 'cogs phải >= 0.')
record_mask('business_rule', 'products', 'cogs_not_above_price', products['cogs'] <= products['price'], 'Snapshot cogs không vượt list price.')
record_mask('business_rule', 'order_items', 'quantity_positive', items['quantity'] > 0, 'quantity phải > 0.')
record_mask('business_rule', 'order_items', 'unit_price_nonnegative', items['unit_price'] >= 0, 'unit_price phải >= 0.')
record_mask('business_rule', 'order_items', 'discount_valid', (items['discount_amount'] >= 0) & (items['discount_amount'] <= items['quantity'] * items['unit_price']), 'discount nằm trong [0, gross].')
record_mask('business_rule', 'payments', 'payment_value_nonnegative', payments['payment_value'] >= 0, 'payment_value phải >= 0.')
record_mask('business_rule', 'payments', 'installments_positive', payments['installments'] > 0, 'installments phải > 0.')
record_mask('business_rule', 'promotions', 'date_order', promotions['end_date'] >= promotions['start_date'], 'end_date không trước start_date.')
record_mask('business_rule', 'promotions', 'discount_value_positive', promotions['discount_value'] > 0, 'discount_value phải > 0.')
record_mask('business_rule', 'promotions', 'percentage_at_most_100', promotions['promo_type'].ne('percentage') | promotions['discount_value'].le(100), 'Promotion percentage không vượt 100.')
record_mask('business_rule', 'promotions', 'min_order_value_nonnegative', promotions['min_order_value'] >= 0, 'min_order_value phải >= 0.')
record_mask('business_rule', 'shipments', 'shipping_fee_nonnegative', shipments['shipping_fee'] >= 0, 'shipping_fee phải >= 0.')
record_mask('business_rule', 'shipments', 'delivery_not_before_ship', shipments['delivery_date'].isna() | (shipments['delivery_date'] >= shipments['ship_date']), 'delivery_date không trước ship_date.')
record_mask('business_rule', 'returns', 'return_quantity_positive', returns['return_quantity'] > 0, 'return_quantity phải > 0.')
record_mask('business_rule', 'returns', 'return_not_above_ordered', returns['return_quantity'] <= returns['ordered_quantity'], 'Không trả quá quantity đã mua.')
record_mask('business_rule', 'returns', 'refund_nonnegative', returns['refund_amount'] >= 0, 'refund_amount phải >= 0.')
record_mask('business_rule', 'reviews', 'rating_1_to_5', reviews['rating'].between(1, 5), 'rating nằm trong 1..5.')
record_check('business_rule', 'reviews', 'review_title_determines_rating', int((reviews.groupby('review_title')['rating'].nunique() > 1).sum()), 'Một review_title phải ánh xạ một rating trong snapshot.')
record_mask('business_rule', 'inventory', 'core_counts_nonnegative', inventory[['stock_on_hand', 'units_received', 'units_sold', 'stockout_days']].ge(0).all(axis=1), 'Các số đo kho không âm.')
record_mask('business_rule', 'inventory', 'stockout_days_0_to_30', inventory['stockout_days'].between(0, 30), 'Dataset dùng tháng quy ước 30 ngày.')
record_mask('business_rule', 'web_traffic', 'traffic_counts_nonnegative', traffic[['sessions', 'unique_visitors', 'page_views']].ge(0).all(axis=1), 'Các số đếm traffic không âm.')
record_mask('business_rule', 'web_traffic', 'visitors_not_above_sessions', traffic['unique_visitors'] <= traffic['sessions'], 'unique_visitors không vượt sessions.')
record_mask('business_rule', 'web_traffic', 'bounce_rate_0_to_1', traffic['bounce_rate'].between(0, 1), 'bounce_rate nằm trong [0,1].')
record_mask('business_rule', 'web_traffic', 'duration_nonnegative', traffic['avg_session_duration_sec'] >= 0, 'Duration không âm.')
record_mask('business_rule', 'sales', 'targets_nonnegative', sales[['Revenue', 'COGS']].ge(0).all(axis=1), 'Revenue và COGS không âm.')
record_mask('business_rule', 'sample_submission', 'forecast_values_nonnegative', forecast[['Revenue', 'COGS']].ge(0).all(axis=1), 'Giá trị dự báo mẫu không âm; đây không phải actual.')

# Cảnh báo có chủ đích: phát hiện nhưng không tự sửa.
record_check('warning', 'products', 'price_below_100', int(products['price'].lt(100).sum()), 'Hai thang giá đáng audit; không tự sửa.', 'WARN')
record_check('warning', 'inventory', 'constant_reorder_flag', int(inventory['reorder_flag'].eq(False).sum()), 'reorder_flag là hằng số và không mang thông tin.', 'WARN')
record_check('warning', 'orders', 'orders_without_shipment', int((~orders['order_id'].isin(shipments['order_id'])).sum()), 'Có thể hợp lệ theo lifecycle; không tự gắn lỗi.', 'WARN')

### Nhận xét và kết luận

- Cell kiểm tra domain của các trường phân loại, 14 nhóm quan hệ FK, tính dương/không âm, thứ tự ngày và các quy tắc như `discount_amount <= quantity × unit_price`, rating trong 1–5 và return không vượt quantity đã mua.
- Cell không in kết quả ngay mà ghi từng check vào `quality_records`; cách này tránh phân tán kết luận và cho phép quality gate tổng hợp ở cell acceptance.
- Các pattern chưa đủ bằng chứng sửa — giá thấp, `reorder_flag` hằng số và order thiếu shipment — được ghi mức `WARN`, không trộn với lỗi contract.

**Kết luận:** cell đã định nghĩa và thực thi lớp kiểm tra nghiệp vụ/toàn vẹn; trạng thái đạt hay không phải được đọc ở quality gate, không suy ra chỉ từ việc cell chạy không in output.

## 6. Đối chiếu chéo và tái tạo chỉ tiêu

Các cột copy được đối chiếu trước khi giữ lại. Payment được so với net order value; `sales.csv` được tái tạo từ orders × order_items × products; năm chỉ báo inventory được tính lại.

In [6]:
customer_geo = customers.merge(geography[['zip', 'city']], on='zip', suffixes=('_customer', '_geography'), validate='many_to_one')
record_check('reconciliation', 'customers', 'city_matches_geography', int(customer_geo['city_customer'].ne(customer_geo['city_geography']).sum()), 'city copy phải khớp geography.')
order_customer = orders.merge(customers[['customer_id', 'zip']], on='customer_id', suffixes=('_order', '_customer'), validate='many_to_one')
record_check('reconciliation', 'orders', 'zip_matches_customer', int(order_customer['zip_order'].ne(order_customer['zip_customer']).sum()), 'zip copy phải khớp customer.')
payment_order = payments.merge(orders[['order_id', 'payment_method']], on='order_id', suffixes=('_payment', '_order'), validate='one_to_one')
record_check('reconciliation', 'payments', 'method_matches_order', int(payment_order['payment_method_payment'].ne(payment_order['payment_method_order']).sum()), 'payment_method copy phải khớp order.')
review_order = reviews.merge(orders[['order_id', 'customer_id', 'order_date']], on='order_id', suffixes=('_review', '_order'), validate='many_to_one')
record_check('reconciliation', 'reviews', 'customer_matches_order', int(review_order['customer_id_review'].ne(review_order['customer_id_order']).sum()), 'customer_id copy phải khớp order.')
record_check('business_rule', 'reviews', 'review_not_before_order', int((review_order['review_date'] < review_order['order_date']).sum()), 'review_date không trước order_date.')
return_order = returns.merge(orders[['order_id', 'order_date']], on='order_id', validate='many_to_one')
record_check('business_rule', 'returns', 'return_not_before_order', int((return_order['return_date'] < return_order['order_date']).sum()), 'return_date không trước order_date.')

inventory_product = inventory.merge(products[['product_id', 'product_name', 'category', 'segment']], on='product_id', suffixes=('_inventory', '_product'), validate='many_to_one')
for column in ['product_name', 'category', 'segment']:
    record_check('reconciliation', 'inventory', f'{column}_matches_product', int(inventory_product[f'{column}_inventory'].ne(inventory_product[f'{column}_product']).sum()), f'{column} copy phải khớp product.')
record_check('reconciliation', 'inventory', 'year_matches_snapshot', int(inventory['year'].ne(inventory['snapshot_date'].dt.year).sum()), 'year phải suy ra từ snapshot_date.')
record_check('reconciliation', 'inventory', 'month_matches_snapshot', int(inventory['month'].ne(inventory['snapshot_date'].dt.month).sum()), 'month phải suy ra từ snapshot_date.')

days_of_supply = (inventory['stock_on_hand'] / (inventory['units_sold'] / 30)).round(1)
fill_rate = (1 - inventory['stockout_days'] / 30).round(4)
stockout_flag = inventory['stockout_days'].gt(0)
sell_through_rate = (inventory['units_sold'] / (inventory['stock_on_hand'] + inventory['units_sold'])).round(4)
overstock_flag = days_of_supply.gt(90)
derived_checks = {
    'days_of_supply': np.isclose(inventory['days_of_supply'], days_of_supply, atol=1e-9),
    'fill_rate': np.isclose(inventory['fill_rate'], fill_rate, atol=1e-9),
    'stockout_flag': inventory['stockout_flag'].eq(stockout_flag),
    'sell_through_rate': np.isclose(inventory['sell_through_rate'], sell_through_rate, atol=1e-9),
    'overstock_flag': inventory['overstock_flag'].eq(overstock_flag),
}
for name, valid in derived_checks.items():
    record_check('reconciliation', 'inventory', f'rebuild_{name}', int((~np.asarray(valid)).sum()), 'Chỉ báo dẫn xuất phải tái tạo được.')

net_by_order = ((items['quantity'] * items['unit_price'] - items['discount_amount']).groupby(items['order_id']).sum())
payment_diff = (payments.set_index('order_id')['payment_value'] - net_by_order).abs()
record_check('reconciliation', 'payments', 'net_order_value_tolerance_0_01', int(payment_diff.gt(0.01).sum()), f'max diff={float(payment_diff.max()):.6f}')
transaction = items.merge(products[['product_id', 'cogs']], on='product_id', validate='many_to_one').merge(orders[['order_id', 'order_date']], on='order_id', validate='many_to_one')
daily = transaction.assign(Revenue=lambda x: x['quantity'] * x['unit_price'], COGS=lambda x: x['quantity'] * x['cogs']).groupby('order_date', as_index=False)[['Revenue', 'COGS']].sum()
rebuilt = sales.merge(daily, left_on='Date', right_on='order_date', suffixes=('_source', '_rebuilt'), validate='one_to_one')
revenue_diff = (rebuilt['Revenue_source'] - rebuilt['Revenue_rebuilt']).abs()
cogs_diff = (rebuilt['COGS_source'] - rebuilt['COGS_rebuilt']).abs()
record_check('reconciliation', 'sales', 'rebuild_revenue_tolerance_0_01', int(revenue_diff.gt(0.01).sum()), f'max diff={float(revenue_diff.max()):.6f}')
record_check('reconciliation', 'sales', 'rebuild_cogs_tolerance_0_01', int(cogs_diff.gt(0.01).sum()), f'max diff={float(cogs_diff.max()):.6f}')
record_check('reconciliation', 'all', 'total_row_count_preserved', abs(sum(len(frame) for frame in cleaned.values()) - TOTAL_EXPECTED_ROWS), 'Tổng output phải giữ 2.960.736 dòng.')

# Cảnh báo signup_date không dùng cho cohort nếu chưa xác minh nguồn.
first_order = orders.groupby('customer_id', as_index=False)['order_date'].min()
signup_check = customers[['customer_id', 'signup_date']].merge(first_order, on='customer_id', how='left', validate='one_to_one')
record_check('warning', 'customers', 'first_order_before_signup', int((signup_check['order_date'] < signup_check['signup_date']).sum()), 'Không tự sửa signup_date; không dùng cho cohort.', 'WARN')
print(f'Payment max diff={float(payment_diff.max()):.6f}; Revenue max diff={float(revenue_diff.max()):.6f}; COGS max diff={float(cogs_diff.max()):.6f}.')

Payment max diff=0.000000; Revenue max diff=0.000000; COGS max diff=0.004999.


### Nhận xét và kết luận

- Net order value tái tạo từ order items khớp payment với sai lệch tối đa **0,000000**.
- Gross Revenue tái tạo từ `orders × order_items` khớp `sales.csv` với sai lệch tối đa **0,000000**.
- COGS tái tạo qua product có sai lệch tối đa **0,004999**, nhỏ hơn dung sai 0,01; đây là sai số làm tròn số thực, không phải mất giao dịch.
- Các cột copy giữa customer/geography, order/customer, payment/order, review/order và inventory/product được đối chiếu; năm chỉ báo inventory cũng được tính lại.

**Kết luận:** các nguồn giao dịch và bảng tổng hợp nhất quán nội bộ đối với payment, Revenue, COGS và inventory. Kết quả này chứng minh khả năng tái tạo trong dataset, không chứng minh dữ liệu phản ánh hoàn hảo thế giới thực.

## 7. Kiểm tra và gắn cờ ngoại lệ

Outlier chỉ là dấu hiệu cần review, không chứng minh dữ liệu sai. Notebook dùng hàng rào Tukey `Q3 + 1,5 × IQR` cho các số đo không âm; riêng `order_items.unit_price` dùng hàng rào hai phía trong từng `product_id` có ít nhất 20 quan sát. Giá trị gốc không bị xóa, clip hoặc winsorize.

In [7]:
outlier_records = []
for dataset, frame in cleaned.items():
    masks = []
    for column in OUTLIER_COLUMNS.get(dataset, []):
        values = pd.to_numeric(frame[column], errors='coerce').astype('float64')
        q1, q3 = values.quantile([0.25, 0.75])
        upper = q3 + 1.5 * (q3 - q1)
        mask = values.gt(upper).fillna(False).to_numpy(dtype=bool)
        flag = f'{column}_outlier'
        frame[flag] = mask
        masks.append(mask)
        outlier_records.append({'dataset': dataset, 'column': column, 'method': 'Tukey upper 1.5 IQR', 'lower_fence': None, 'upper_fence': float(upper), 'outlier_rows': int(mask.sum()), 'outlier_pct': float(mask.mean())})

    if dataset == 'order_items':
        counts = frame.groupby('product_id')['unit_price'].count()
        quantiles = frame.groupby('product_id')['unit_price'].quantile([0.25, 0.75]).unstack()
        quantiles.columns = ['q1', 'q3']
        quantiles['lower'] = quantiles['q1'] - 1.5 * (quantiles['q3'] - quantiles['q1'])
        quantiles['upper'] = quantiles['q3'] + 1.5 * (quantiles['q3'] - quantiles['q1'])
        eligible = counts.ge(20)
        lower = frame['product_id'].map(quantiles['lower'])
        upper = frame['product_id'].map(quantiles['upper'])
        eligible_rows = frame['product_id'].map(eligible).fillna(False)
        unit_price_mask = (eligible_rows & (frame['unit_price'].lt(lower) | frame['unit_price'].gt(upper))).to_numpy(dtype=bool)
        frame['unit_price_outlier'] = unit_price_mask
        masks.append(unit_price_mask)
        outlier_records.append({'dataset': dataset, 'column': 'unit_price', 'method': 'Tukey two-sided within product_id; n>=20', 'lower_fence': None, 'upper_fence': None, 'outlier_rows': int(unit_price_mask.sum()), 'outlier_pct': float(unit_price_mask.mean())})

    frame['is_outlier'] = np.logical_or.reduce(masks) if masks else np.zeros(len(frame), dtype=bool)
    if masks:
        detailed_or = np.logical_or.reduce(masks)
        record_check('outlier', dataset, 'is_outlier_equals_detailed_flags', int((frame['is_outlier'].to_numpy() != detailed_or).sum()), 'is_outlier phải là OR của cờ chi tiết.')

outlier_summary = pd.DataFrame(outlier_records)
display(outlier_summary)
display(pd.DataFrame({'dataset': list(cleaned), 'rows': [len(frame) for frame in cleaned.values()], 'outlier_rows': [int(frame['is_outlier'].sum()) for frame in cleaned.values()]}))

,dataset,column,method,lower_fence,upper_fence,outlier_rows,outlier_pct
0,inventory,stock_on_hand,Tukey upper 1.5 IQR,None,5.025000e+02,6432,0.106761
1,inventory,units_received,Tukey upper 1.5 IQR,None,4.450000e+01,6392,0.106097
2,inventory,units_sold,Tukey upper 1.5 IQR,None,3.700000e+01,6388,0.106030
3,inventory,stockout_days,Tukey upper 1.5 IQR,None,5.000000e+00,724,0.012017
4,inventory,days_of_supply,Tukey upper 1.5 IQR,None,1.563750e+03,7455,0.123741
5,inventory,fill_rate,Tukey upper 1.5 IQR,None,1.100050e+00,0,0.000000
6,inventory,sell_through_rate,Tukey upper 1.5 IQR,None,5.321000e-01,1284,0.021312
7,order_items,quantity,Tukey upper 1.5 IQR,None,1.200000e+01,0,0.000000
8,order_items,discount_amount,Tukey upper 1.5 IQR,None,2.419075e+03,105767,0.147994
9,order_items,unit_price,Tukey two-sided within product_id; n>=20,None,NaN,23257,0.032542


,dataset,rows,outlier_rows
0,customers,121930,0
1,geography,39948,0
2,inventory,60247,16512
3,order_items,714669,128679
4,orders,646945,0
5,payments,646945,30219
6,products,2412,47
7,promotions,50,5
8,returns,39939,2778
9,reviews,113551,0


### Nhận xét và kết luận

- Có **27 quy tắc outlier** và **254.527 dòng** được gắn ít nhất một cờ, tương đương khoảng **8,60%** toàn dataset. Tổng dòng unique này không phải tổng các cờ chi tiết vì một dòng có thể vi phạm nhiều ngưỡng.
- Số dòng được gắn cờ tập trung ở `order_items` (**128.679**), `shipments` (**76.050**), `payments` (**30.219**) và `inventory` (**16.512**).
- Trong `order_items`, discount_amount có **105.767** cờ phía trên và unit_price có **23.257** cờ theo phân phối từng product; quantity không có outlier theo ngưỡng hiện tại.
- `sample_submission` có **20** dòng được gắn cờ, nhưng đây là giá trị dự báo mẫu chứ không phải actual; cần diễn giải riêng khi review.

**Kết luận:** outlier đã được định vị để ưu tiên kiểm tra nhưng không có dòng nào bị xóa, clip hoặc winsorize. Cờ Tukey là tín hiệu thống kê phụ thuộc snapshot, không đồng nghĩa lỗi nghiệp vụ.

## 8. Acceptance checks

Notebook chỉ ghi output khi tất cả check mức FAIL đều có 0 vi phạm. WARN được giữ lại để người dùng review và không làm thay đổi giá trị nguồn.

In [8]:
quality_summary = pd.DataFrame(quality_records)
failures = quality_summary.query("severity == 'FAIL' and violations > 0")
warnings = quality_summary.query("severity == 'WARN' and violations > 0")
display(quality_summary.groupby(['category', 'status'], as_index=False).size())
display(warnings)
if len(failures):
    display(failures)
    raise DataQualityError(f'Có {len(failures)} acceptance check không đạt.')
assert len(cleaned) == 14
assert sum(len(frame) for frame in cleaned.values()) == TOTAL_EXPECTED_ROWS
assert len(promo_links) == 276_522
print(f'ĐẠT: {len(quality_summary) - len(warnings)} check không có lỗi; giữ {len(warnings)} cảnh báo để review.')

,category,status,size
0,business_rule,PASS,45
1,duplicate,PASS,30
2,integrity,PASS,16
3,missing,PASS,14
4,outlier,PASS,10
5,reconciliation,PASS,20
6,warning,WARN,4


,category,dataset,check,severity,violations,status,details
105,warning,products,price_below_100,WARN,654,WARN,Hai thang giá đáng audit; không tự sửa.
106,warning,inventory,constant_reorder_flag,WARN,60247,WARN,reorder_flag là hằng số và không mang thông tin.
107,warning,orders,orders_without_shipment,WARN,80878,WARN,Có thể hợp lệ theo lifecycle; không tự gắn lỗi.
128,warning,customers,first_order_before_signup,WARN,80623,WARN,Không tự sửa signup_date; không dùng cho cohort.


ĐẠT: 135 check không có lỗi; giữ 4 cảnh báo để review.


### Nhận xét và kết luận

- Quality gate tổng hợp **139 check**: **135 PASS**, **4 WARN** và **0 hard failure**. Vì không có check mức FAIL vi phạm, notebook được phép ghi output.
- Bốn cảnh báo được giữ lại gồm: **654** sản phẩm có `price < 100`, **60.247** dòng inventory có `reorder_flag = 0`, **80.878** order không có shipment và **80.623** customer có first order trước signup date.
- WARN không được hiểu là 4 lỗi cần tự động sửa; mỗi cảnh báo cần nguồn nghiệp vụ hoặc phân tích lifecycle bổ sung trước khi quyết định.

**Kết luận:** snapshot đạt toàn bộ điều kiện bắt buộc của pipeline và đủ điều kiện tạo lớp cleaned; các giới hạn đã biết vẫn được công bố thay vì che giấu.

## 9. Ghi Parquet và đọc lại

Mỗi bảng được ghi qua file tạm, đọc lại để so schema và toàn bộ giá trị, rồi mới atomic replace. Ba báo cáo `profile_summary`, `quality_summary` và `outlier_summary` cũng được lưu cùng output.

In [9]:
def write_table_atomic(frame, output_path, extra_metadata=None):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(frame, preserve_index=False)
    metadata = dict(table.schema.metadata or {})
    metadata.update({b'cleaning_version': CLEANING_VERSION.encode(), b'row_count': str(len(frame)).encode()})
    if extra_metadata:
        metadata.update({str(key).encode(): str(value).encode() for key, value in extra_metadata.items()})
    table = table.replace_schema_metadata(metadata)
    descriptor, temporary_name = tempfile.mkstemp(prefix=f'.{output_path.stem}.', suffix='.tmp.parquet', dir=output_path.parent)
    os.close(descriptor)
    temporary_path = Path(temporary_name)
    try:
        pq.write_table(table, temporary_path, compression='zstd')
        actual = pq.read_table(temporary_path, memory_map=False)
        schema_matches = (
            actual.schema.names == table.schema.names
            and actual.schema.metadata == table.schema.metadata
            and all(a.type == e.type and a.nullable == e.nullable for a, e in zip(actual.schema, table.schema))
        )
        if not schema_matches:
            raise DataQualityError(f'{output_path.name}: schema/metadata đọc lại không khớp.')
        if not actual.equals(table):
            raise DataQualityError(f'{output_path.name}: giá trị đọc lại không khớp.')
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()
    return output_path.stat().st_size

write_records = []
for dataset, frame in cleaned.items():
    output_path = OUTPUT_DIR / f'{dataset}.parquet'
    size = write_table_atomic(frame, output_path, {'source_file': f'{dataset}.csv', 'source_sha256': source_hashes[dataset], 'grain_preserved': 'true'})
    write_records.append({'output': output_path.as_posix(), 'rows': len(frame), 'columns': len(frame.columns), 'size_mib': size / 1024**2})

for report_name, report in [('profile_summary', profile_summary), ('quality_summary', quality_summary), ('outlier_summary', outlier_summary)]:
    output_path = OUTPUT_DIR / f'_{report_name}.parquet'
    size = write_table_atomic(report, output_path, {'report': report_name})
    write_records.append({'output': output_path.as_posix(), 'rows': len(report), 'columns': len(report.columns), 'size_mib': size / 1024**2})

write_summary = pd.DataFrame(write_records)
display(write_summary)
assert len(list(OUTPUT_DIR.glob('*.parquet'))) == 17
assert sum(pq.read_metadata(OUTPUT_DIR / f'{dataset}.parquet').num_rows for dataset in cleaned) == TOTAL_EXPECTED_ROWS
print(f'ĐẠT ghi và đọc lại 14 bảng + 3 báo cáo tại {OUTPUT_DIR}.')

,output,rows,columns,size_mib
0,data/silver/full_dataset_notebook/customers.pa...,121930,9,1.264461
1,data/silver/full_dataset_notebook/geography.pa...,39948,6,0.256837
2,data/silver/full_dataset_notebook/inventory.pa...,60247,26,0.663166
3,data/silver/full_dataset_notebook/order_items....,714669,13,7.577116
4,data/silver/full_dataset_notebook/orders.parquet,646945,10,5.789268
5,data/silver/full_dataset_notebook/payments.par...,646945,7,5.283930
6,data/silver/full_dataset_notebook/products.par...,2412,12,0.063808
7,data/silver/full_dataset_notebook/promotions.p...,50,14,0.010466
8,data/silver/full_dataset_notebook/returns.parquet,39939,14,0.839615
9,data/silver/full_dataset_notebook/reviews.parquet,113551,11,1.714856


ĐẠT ghi và đọc lại 14 bảng + 3 báo cáo tại data\silver\full_dataset_notebook.


### Nhận xét và kết luận

- Cell đã ghi thành công **14 bảng cleaned và 3 báo cáo Parquet**: profile, quality và outlier. Tổng row count của 14 bảng vẫn là **2.960.736**, nên không có bản ghi nguồn bị mất.
- `order_items.parquet` là bảng lớn nhất, khoảng **7,58 MiB**; toàn bộ 17 file chiếm khoảng **27,4 MiB** với nén Zstandard.
- Mỗi file được ghi qua đường dẫn tạm, đọc lại để so schema/metadata và toàn bộ giá trị rồi mới atomic replace. Vì vậy output cũ không bị thay bằng file ghi dở khi validation thất bại.
- CSV nguồn không bị ghi đè; output nằm riêng tại `data/silver/full_dataset_notebook/`.

**Kết luận:** quy trình đã hoàn tất cả biến đổi lẫn kiểm chứng vật lý đầu ra, tạo bộ Parquet có thể dùng cho feature engineering và mô hình hóa.

## Kết luận

Notebook đã chạy đủ chuỗi:

1. profiling toàn bộ cột;
2. kiểm tra missing bắt buộc và chuẩn hóa null optional;
3. kiểm tra duplicate, giữ đúng các dòng hàng lặp hợp lệ;
4. chuẩn hóa string, integer, number, boolean và date;
5. gắn cờ outlier nhưng không thay giá trị;
6. kiểm tra domain và business rule;
7. kiểm tra PK/FK và occurrence mapping;
8. đối chiếu payment, sales, inventory và các cột copy;
9. ghi–đọc lại 14 Parquet mà không ghi đè dữ liệu nguồn.

Các cảnh báo không đủ bằng chứng sửa được giữ nguyên và công bố trong `_quality_summary.parquet`.